# RQ3: Exploratory Correlation Analysis of Demographic and Dispositional Variables

This notebook investigates bivariate relationships between demographic characteristics, dispositional traits, and measures of trust, system understanding, and mental workload. The Linear Mixed Models can be found in the accompanying R script.

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

from scipy.stats import spearmanr
from matplotlib.colors import LinearSegmentedColormap
from statsmodels.stats.multitest import multipletests

## Load data

In [ ]:
df = pd.read_csv("../DATA/demo.csv")
blocks = pd.read_csv("../DATA/blocks.csv")

df = (
    blocks
    .merge(df, on="participant_id", how="left")
    .replace({
        "condition": {
            "Nelo": "No explanation",
            "Coda": "Push explanation",
            "Lumo": "Pull explanation",
        }
    })
)

df["block"] = df["block"].astype("category")

LABELS = {
    "trust": "Trust Score",
    "understanding": "System Understanding Score",
    "nasa": "Workload",
    "tri_score": "Technology Readiness Index",
    "nfc_score": "Need for Cognition",
    "bfi_extraversion": "Extraversion",
    "bfi_agreeableness": "Agreeableness",
    "bfi_conscientiousness": "Conscientiousness",
    "bfi_neuroticism": "Neuroticism",
    "bfi_openness": "Openness",
}

BFI_COLS = [
    "bfi_extraversion",
    "bfi_agreeableness",
    "bfi_conscientiousness",
    "bfi_neuroticism",
    "bfi_openness",
]

PREDICTORS = [
    "tri_score",
    "nfc_score",
    *BFI_COLS,
    "driving_frequency_num",
    "drivers_license_num",
    "automation_experience_num",
    "age",
]

OUTCOMES = ["trust", "understanding", "nasa"]

sns.set_theme(style="whitegrid", font_scale=1.4)


MAPS = {
    "driving_frequency": {
        "never": 0,
        "rarely": 1,
        "several times a week": 2,
        "daily": 3,
    },
    "automation_experience": {
        "no": 0,
        "yes": 1,
        "not sure": 0,
    },
    "drivers_license": {
        "no": 0,
        "yes": 1,
    },
    "vr_experience": {
        "no": 0,
        "yes": 1,
    },
}

for column, mapping in MAPS.items():
    df[f"{column}_num"] = df[column].str.lower().map(mapping)

df["age"] = 2026 - df["birth_year"]

df_corr = pd.DataFrame(
    [
        {
            "predictor": predictor,
            "outcome": outcome,
            "correlation": spearmanr(df[predictor], df[outcome]).statistic,
            "p_value": spearmanr(df[predictor], df[outcome]).pvalue,
        }
        for predictor in PREDICTORS
        for outcome in OUTCOMES
    ]
)

df_corr

In [ ]:
VAR_LABELS = {
    "trust": "Trust",
    "understanding": "System Understanding",
    "nasa": "Mental Workload",
    "technology_readiness": "Technology Readiness",
    "tri": "Technology Readiness",
    "tri_score": "Technology Readiness",
    "nfc": "Need for Cognition",
    "nfc_score": "Need for Cognition",
    "need_for_cognition": "Need for Cognition",
    "bfi_extraversion": "Extraversion",
    "bfi_agreeableness": "Agreeableness",
    "bfi_conscientiousness": "Conscientiousness",
    "bfi_neuroticism": "Negative Emotionality",
    "bfi_openness": "Openness",
    "age": "Age",
    "automation_experience": "Automation Experience",
    "automation_familiarity": "Automation Familiarity",
    "driving_frequency_num": "Driving Frequency",
    "automation_experience_num": "Automation Experience",
    "drivers_license_num": "Driver's License",
    "vr_experience_num": "VR Experience",
}

cols = PREDICTORS + OUTCOMES
data = df[cols]

n = len(cols)
rho = np.eye(n)
pvals = np.zeros((n, n))
lower = np.tril_indices(n, k=-1)

for i, j in zip(*lower):
    r, p = stats.spearmanr(data.iloc[:, i], data.iloc[:, j], nan_policy="omit")

    rho[i, j] = rho[j, i] = r
    pvals[i, j] = pvals[j, i] = p

labels = [VAR_LABELS.get(c, c) for c in cols]

## Full Spearman Matrix

In [ ]:
lower_idx = np.tril_indices(n, k=-1)
p_vals_flat = pvals[lower_idx]

reject, p_corrected_flat, _, _ = multipletests(
    p_vals_flat,
    method="fdr_bh"
)

sig_mat = np.zeros((n, n), dtype=bool)
sig_mat[lower_idx] = reject


def stars(p):
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""

annot = np.full((n, n), "", dtype=object)

for (i, j), p_corr in zip(zip(*lower_idx), p_corrected_flat):
    value = f"{rho[i, j]:.2f}"

    if sig_mat[i, j]:
        annot[i, j] = f"{value}\n{stars(p_corr)}"
    else:
        annot[i, j] = value

annot_df = pd.DataFrame(
    annot,
    index=labels,
    columns=labels,
)


corr_df = pd.DataFrame(rho, index=labels, columns=labels)

upper = np.triu(np.ones((n, n), dtype=bool))

sig_lower = np.zeros((n, n), dtype=bool)
sig_lower[lower_idx] = sig_mat[lower_idx]

fig, ax = plt.subplots(figsize=(9, 7), dpi=300)

cmap = LinearSegmentedColormap.from_list(
    "custom_diverging",
    ["#491d8b", "#f7f7f7", "#004144"],
)

mask = np.triu(np.ones((n, n), dtype=bool))

sns.heatmap(
    corr_df,
    mask=mask,
    cmap=cmap,
    center=0,
    vmin=-1,
    vmax=1,
    annot=annot_df,
    fmt="",
    annot_kws={"size": 9},
    square=True,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={
        "label": "Spearman ρ",
        "shrink": 0.8,
    },
    ax=ax,
)


cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=10)
cbar.set_label("Spearman ρ", size=10)

ax.tick_params(axis="x", rotation=45, labelsize=12)
ax.tick_params(axis="y", rotation=0, labelsize=12)

plt.setp(ax.get_xticklabels(), ha="right")

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

### Spearman Matrix Masked Version

In [ ]:
annot = np.full((n, n), "", dtype=object)

for (i, j), p_corr in zip(zip(*lower_idx), p_corrected_flat):
    if sig_mat[i, j]:
        annot[i, j] = f"{rho[i, j]:.2f}\n{stars(p_corr)}"

corr_df = pd.DataFrame(rho, index=labels, columns=labels)
annot_df = pd.DataFrame(annot, index=labels, columns=labels)


upper = np.triu(np.ones((n, n), dtype=bool))

sig_lower = np.zeros((n, n), dtype=bool)
sig_lower[lower_idx] = sig_mat[lower_idx]

mask_colour = upper | ~sig_lower   
mask_grey   = upper | sig_lower    

fig, ax = plt.subplots(figsize=(9, 7), dpi=300)

cmap = LinearSegmentedColormap.from_list(
    "custom_diverging",
    ["#491d8b", "#f7f7f7", "#004144"],
)

grey = LinearSegmentedColormap.from_list(
    "grey",
    ["#e0e0e0", "#e0e0e0"],
)

sns.heatmap(
    np.zeros((n, n)),
    mask=mask_grey,
    cmap=grey,
    cbar=False,
    square=True,
    linewidths=0.5,
    linecolor="white",
    ax=ax,
)

sns.heatmap(
    corr_df,
    mask=mask_colour,
    cmap=cmap,
    center=0,
    vmin=-1,
    vmax=1,
    annot=annot_df,
    fmt="",
    annot_kws={"size": 9},
    square=True,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={
        "label": "Spearman ρ",
        "shrink": 0.8,
    },
    ax=ax,
)

cbar = ax.collections[1].colorbar
cbar.ax.tick_params(labelsize=10)
cbar.set_label("Spearman ρ", size=10)

ax.tick_params(axis="x", rotation=45, labelsize=12)
ax.tick_params(axis="y", rotation=0, labelsize=12)

plt.setp(ax.get_xticklabels(), ha="right")

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.savefig("plots/correlation_heatmap.pdf", dpi=300, bbox_inches="tight")
plt.show()